# LJP Criminal 데이터셋 전처리 가이드

이 노트북은 `lbox/lbox_open`의 `ljp_criminal` 데이터셋을 전처리하는 방법을 설명합니다.

## 데이터셋 개요
- **총 샘플 수**: 8,400건
- **사건 유형**: 강제추행, 공무집행방해, 교통사고처리특례법위반, 도로교통법위반(음주운전), 사기, 상해, 폭행
- **목적**: 범죄 사실(facts)을 입력받아 형량(label)을 예측하는 Legal Judgment Prediction 작업

In [ ]:
# 필요한 라이브러리 설치
!pip install datasets transformers pandas scikit-learn -q

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
from collections import Counter
import numpy as np

# 데이터셋 로드
ds = load_dataset('lbox/lbox_open', 'ljp_criminal', split='train')
print(f"총 샘플 수: {len(ds)}")
print(f"컬럼: {ds.column_names}")

---

## 1. Facts 컬럼 전처리

### 📊 현재 상태
- 평균 길이: 391.6자
- 범위: 77자 ~ 5,020자 (매우 큰 편차)
- 결측치: 없음

### 🤔 왜 이렇게 전처리할까요?

**1) 개인정보 익명화가 필요한 이유:**
- 법률 문서에는 "피해자 B", "C역" 같은 익명화된 부분이 있습니다
- 하지만 일부 샘플에는 실명이나 구체적 지명이 남아있을 수 있습니다
- 모델이 특정 인명이나 지명을 학습하면 오버피팅될 수 있습니다
- **해결책**: 대문자 단일 알파벳 패턴을 [NAME], [PLACE] 같은 특수 토큰으로 통일

**2) 날짜/시간 정규화가 필요한 이유:**
- "2020. 9. 25. 23:45경" 같은 구체적 시간은 판결에 큰 영향을 주지 않습니다
- 다양한 날짜 형식이 있어 토큰 수가 불필요하게 증가합니다
- **해결책**: [DATE], [TIME] 토큰으로 통일하거나, 시간대(새벽/오전/오후/야간)로 변환

**3) 공백 및 특수문자 정리가 필요한 이유:**
- 중복 공백, 불필요한 줄바꿈이 있으면 토큰화가 비효율적입니다
- **해결책**: 연속된 공백을 하나로, 앞뒤 공백 제거

**4) 길이 제한이 필요한 이유:**
- BERT 같은 모델은 최대 512 토큰 제한이 있습니다
- 5,020자짜리 텍스트는 잘리거나 처리 불가능합니다
- **해결책**: 첫 부분만 사용하거나(범죄 사실의 핵심은 앞부분에 많음), sliding window 방식 사용

In [ ]:
def preprocess_facts(text, max_length=1000):
    """
    Facts 컬럼 전처리 함수
    
    처리 순서:
    1. 날짜 정규화 (2020. 9. 25. -> [DATE])
    2. 시간 정규화 (23:45경 -> [TIME])
    3. 단일 대문자 익명화 (B, C -> [NAME])
    4. 공백 정리
    5. 길이 제한
    """
    if not text:
        return ""
    
    # 1. 날짜 패턴 정규화 (예: 2020. 9. 25.)
    # 이유: 구체적 날짜는 형량 예측에 큰 영향을 주지 않으며, 토큰 수를 줄일 수 있습니다
    text = re.sub(r'\d{4}\. \d{1,2}\. \d{1,2}\.', '[DATE]', text)
    
    # 2. 시간 패턴 정규화 (예: 23:45경, 08:00경)
    # 이유: 시간대(새벽/야간)는 의미가 있을 수 있지만, 정확한 분단위는 불필요합니다
    text = re.sub(r'\d{1,2}:\d{2}경?', '[TIME]', text)
    
    # 3. 단일 대문자 알파벳 익명화 (B, C역, D등) - 인명/지명
    # 이유: 개인정보 보호 + 모델이 특정 문자에 오버피팅되는 것 방지
    text = re.sub(r'\s([A-Z])(?=\s|역|동|가|건물|오피스텔|아파트)', r' [PLACE]', text)
    text = re.sub(r'피해자\s+([A-Z])\(', r'피해자 [NAME](', text)
    text = re.sub(r'피고인\s+([A-Z])\(', r'피고인 [NAME](', text)
    
    # 4. 숫자 뒤 "원" 처리 (금액)
    # 이유: 구체적 금액은 범위로 표현하는 것이 더 일반화에 좋습니다
    def amount_to_range(match):
        amount = int(match.group(1).replace(',', ''))
        if amount < 100000:
            return '[10만원미만]'
        elif amount < 1000000:
            return '[100만원미만]'
        elif amount < 10000000:
            return '[1000만원미만]'
        else:
            return '[1000만원이상]'
    
    text = re.sub(r'([\d,]+)원', amount_to_range, text)
    
    # 5. 연속된 공백을 하나로
    # 이유: 토큰화 효율성 향상
    text = re.sub(r'\s+', ' ', text)
    
    # 6. 앞뒤 공백 제거
    text = text.strip()
    
    # 7. 길이 제한 (너무 길면 앞부분만)
    # 이유: BERT 계열 모델의 토큰 제한 (512) 고려
    # 범죄 사실의 핵심은 대부분 앞부분에 기술됩니다
    if len(text) > max_length:
        text = text[:max_length]
    
    return text

# 테스트
sample_text = ds[0]['facts']
print("원본:")
print(sample_text)
print("\n전처리 후:")
print(preprocess_facts(sample_text))
print(f"\n길이: {len(sample_text)} -> {len(preprocess_facts(sample_text))}")

---

## 2. Label 컬럼 전처리

### 📊 현재 상태
- 유니크 라벨: 73개 (징역 6월, 벌금 3000000원 등)
- 구조: `{'text': '징역 6월', 'fine_lv': 0, 'imprisonment_with_labor_lv': 2, 'imprisonment_without_labor_lv': 0}`
- 가장 많은 라벨: 징역 6월(1,370건), 징역 12월(999건)

### 🤔 왜 이렇게 전처리할까요?

**문제점 분석:**
1. **클래스 불균형**: 73개 라벨 중 일부는 1~2개뿐 → 학습 어려움
2. **세밀한 분류**: "징역 6월"과 "징역 8월"을 구분하기는 어렵고 실용성도 낮음
3. **복합 정보**: 형량 타입(징역/벌금/금고) + 양(6월/12월) 두 정보가 섞여있음

**전처리 전략 3가지:**

### 전략 A: 형량 범위로 그룹화 (권장 ⭐)
- **이유**: 실무에서는 정확한 개월수보다 "단기형(6월 미만)", "중기형(6월~1년)", "장기형(1년 이상)" 구분이 더 중요
- **장점**: 클래스 수 감소(73 → 10~15개), 불균형 완화, 예측 안정성 향상
- **단점**: 세밀한 예측 불가

### 전략 B: 멀티태스크 학습
- **이유**: 형벌 타입(징역/벌금/금고)과 형량(짧음/보통/김)을 별도 문제로 분리
- **장점**: 각 태스크가 단순해져 학습이 쉬워짐
- **단점**: 모델 구조 복잡, 구현 난이도 높음

### 전략 C: 회귀(Regression) 문제로 변환
- **이유**: 형량을 "개월 수"나 "금액"으로 변환하여 연속값 예측
- **장점**: 세밀한 예측 가능, 클래스 불균형 해소
- **단점**: 형벌 타입(징역/벌금) 별도 처리 필요

In [ ]:
# 전략 A: 형량 범위로 그룹화 (가장 실용적)

def label_to_category(label_dict):
    """
    Label을 카테고리로 변환
    
    범주 정의:
    - 벌금형은 금액 구간으로
    - 징역/금고는 기간 구간으로
    
    이렇게 하는 이유:
    1. 법률 실무에서 양형 기준표도 구간으로 되어있습니다
    2. "징역 6월"과 "징역 8월"을 정확히 맞추는 것은 판사도 어렵습니다
    3. 구간 예측이 더 안정적이고 실용적입니다
    """
    text = label_dict['text']
    
    # 벌금형
    if '벌금' in text:
        # 금액 추출
        amount_match = re.search(r'([\d]+)원', text)
        if amount_match:
            amount = int(amount_match.group(1))
            if amount <= 1000000:
                return '벌금_100만원이하'
            elif amount <= 3000000:
                return '벌금_300만원이하'
            elif amount <= 5000000:
                return '벌금_500만원이하'
            else:
                return '벌금_500만원초과'
    
    # 징역형
    elif '징역' in text:
        # 기간 추출
        if '년' in text:
            years = re.search(r'(\d+)년', text)
            if years:
                y = int(years.group(1))
                if y >= 2:
                    return '징역_2년이상'
                else:
                    return '징역_1년이상_2년미만'
        else:
            months = re.search(r'(\d+)월', text)
            if months:
                m = int(months.group(1))
                if m <= 4:
                    return '징역_4월이하'
                elif m <= 6:
                    return '징역_6월이하'
                elif m <= 10:
                    return '징역_10월이하'
                else:
                    return '징역_10월초과'
    
    # 금고형
    elif '금고' in text:
        if '년' in text:
            return '금고_1년이상'
        else:
            return '금고_1년미만'
    
    return '기타'

# 테스트 및 분포 확인
labels_categorized = [label_to_category(sample['label']) for sample in ds]
label_dist = Counter(labels_categorized)

print("범주화된 라벨 분포:")
for label, count in sorted(label_dist.items(), key=lambda x: -x[1]):
    print(f"  {label}: {count}건 ({count/len(ds)*100:.1f}%)")

print(f"\n총 카테고리 수: {len(label_dist)}개 (원래 73개에서 감소)")

In [ ]:
# 전략 B: 멀티태스크 학습용 라벨 생성

def label_to_multitask(label_dict):
    """
    Label을 멀티태스크로 분리
    
    Task 1: 형벌 타입 분류 (징역 / 벌금 / 금고)
    Task 2: 형량 수준 분류 (경미 / 보통 / 중함 / 매우중함)
    
    이유:
    - 형벌 타입과 양은 서로 다른 요인에 의해 결정됩니다
    - 타입: 범죄의 성질
    - 양: 범죄의 경중, 전과, 피해 규모 등
    """
    text = label_dict['text']
    
    # Task 1: 형벌 타입
    if '벌금' in text:
        penalty_type = '벌금'
    elif '징역' in text:
        penalty_type = '징역'
    elif '금고' in text:
        penalty_type = '금고'
    else:
        penalty_type = '기타'
    
    # Task 2: 형량 수준 (레벨 값 활용)
    fine_lv = label_dict['fine_lv']
    imp_labor_lv = label_dict['imprisonment_with_labor_lv']
    imp_no_labor_lv = label_dict['imprisonment_without_labor_lv']
    
    max_lv = max(fine_lv, imp_labor_lv, imp_no_labor_lv)
    
    if max_lv == 0:
        severity = '없음'
    elif max_lv == 1:
        severity = '경미'
    elif max_lv == 2:
        severity = '보통'
    elif max_lv == 3:
        severity = '중함'
    else:
        severity = '매우중함'
    
    return {
        'penalty_type': penalty_type,
        'severity': severity
    }

# 테스트
sample_labels = [label_to_multitask(sample['label']) for sample in ds[:10]]
print("멀티태스크 라벨 예시 (첫 10개):")
for i, label in enumerate(sample_labels):
    print(f"  {i+1}. 타입: {label['penalty_type']}, 수준: {label['severity']}")

# 분포 확인
all_multitask = [label_to_multitask(sample['label']) for sample in ds]
types = Counter([l['penalty_type'] for l in all_multitask])
severities = Counter([l['severity'] for l in all_multitask])

print("\n형벌 타입 분포:")
for t, c in types.items():
    print(f"  {t}: {c}건")

print("\n형량 수준 분포:")
for s, c in severities.items():
    print(f"  {s}: {c}건")

In [ ]:
# 전략 C: 회귀 문제로 변환

def label_to_regression(label_dict):
    """
    Label을 회귀값으로 변환
    
    변환 방식:
    1. 모든 형량을 '개월 수' 단위로 통일
    2. 벌금은 100만원 = 1개월로 환산 (예: 500만원 = 5개월)
    
    이유:
    - 실제 법률에서도 벌금 미납시 환형유치(노역장)로 전환
    - 이 비율이 대략 100만원당 1개월 정도입니다
    - 단일 스케일로 통일하면 회귀 모델 학습이 쉬워집니다
    """
    text = label_dict['text']
    
    # 징역/금고형
    if '년' in text:
        years = re.search(r'(\d+)년', text)
        months = re.search(r'(\d+)월', text)
        
        total_months = 0
        if years:
            total_months += int(years.group(1)) * 12
        if months:
            total_months += int(months.group(1))
        
        return {
            'type': '징역' if '징역' in text else '금고',
            'months_equivalent': total_months
        }
    
    elif '월' in text:
        months = re.search(r'(\d+)월', text)
        if months:
            return {
                'type': '징역' if '징역' in text else '금고',
                'months_equivalent': int(months.group(1))
            }
    
    # 벌금형
    elif '벌금' in text:
        amount = re.search(r'([\d]+)원', text)
        if amount:
            # 100만원 = 1개월로 환산
            months_equiv = int(amount.group(1)) / 1000000
            return {
                'type': '벌금',
                'months_equivalent': round(months_equiv, 1)
            }
    
    return {'type': '기타', 'months_equivalent': 0}

# 테스트
regression_labels = [label_to_regression(sample['label']) for sample in ds]

print("회귀 라벨 예시 (첫 10개):")
for i in range(10):
    orig = ds[i]['label']['text']
    reg = regression_labels[i]
    print(f"  {orig} -> {reg['type']}, {reg['months_equivalent']}개월 상당")

# 통계
months_values = [r['months_equivalent'] for r in regression_labels]
print(f"\n개월 수 통계:")
print(f"  평균: {np.mean(months_values):.1f}개월")
print(f"  중앙값: {np.median(months_values):.1f}개월")
print(f"  최소: {np.min(months_values):.1f}개월")
print(f"  최대: {np.max(months_values):.1f}개월")

---

## 3. Ruling 컬럼 전처리

### 📊 현재 상태
- 평균 길이: 82.9자
- 내용: 판결 주문 ("피고인을 징역 6월에 처한다. 다만, 집행유예...")
- 형벌 유형: 징역(4,238건), 공백(3,458건), 금고(704건)

### 🤔 왜 이렇게 전처리할까요?

**Ruling의 역할:**
- **Label과 거의 동일한 정보**: Ruling에서 형량 부분만 추출하면 Label과 같습니다
- **추가 정보**: 집행유예, 사회봉사, 치료강의 등이 포함될 수 있습니다

**사용 시나리오별 전처리:**

### 시나리오 1: Ruling을 사용하지 않음 (권장 ⭐)
- **이유**: Label과 중복되는 정보이고, 모델의 입력(facts)과 출력(label)만으로 충분합니다
- **장점**: 데이터 누수(data leakage) 방지, 모델 단순화

### 시나리오 2: 집행유예 예측에 활용
- **이유**: "집행유예 여부"는 Label과 별개의 중요한 정보입니다
- **방법**: "집행유예" 키워드 존재 여부를 별도 이진 레이블로 추출

### 시나리오 3: 부가처분 예측
- **이유**: 사회봉사, 치료강의 등은 재범 방지에 중요한 정보입니다
- **방법**: 부가처분 종류와 시간을 추출

In [ ]:
# 시나리오 2 & 3: Ruling에서 추가 정보 추출

def extract_ruling_features(ruling_dict):
    """
    Ruling에서 Label에 없는 추가 정보 추출
    
    추출 항목:
    1. 집행유예 여부 및 기간
    2. 사회봉사 시간
    3. 치료강의 시간
    4. 취업제한
    
    왜 이 정보들이 중요한가?
    - 집행유예: 실제 수감 여부 결정 (사회적 영향 큼)
    - 사회봉사/치료강의: 교화 가능성 판단
    - 취업제한: 재범 방지
    """
    text = ruling_dict['text']
    
    features = {
        'has_suspension': False,  # 집행유예 여부
        'suspension_years': 0,    # 집행유예 기간
        'social_service_hours': 0,  # 사회봉사 시간
        'treatment_hours': 0,      # 치료강의 시간
        'employment_restriction': False  # 취업제한
    }
    
    # 집행유예
    if '집행유예' in text or '집행을 유예' in text:
        features['has_suspension'] = True
        # 기간 추출 (예: "2년간")
        suspension_match = re.search(r'(\d+)년간?\s*위\s*형의\s*집행', text)
        if suspension_match:
            features['suspension_years'] = int(suspension_match.group(1))
    
    # 사회봉사
    social_match = re.search(r'(\d+)시간의\s*사회봉사', text)
    if social_match:
        features['social_service_hours'] = int(social_match.group(1))
    
    # 치료강의 (성폭력, 약물 등)
    treatment_match = re.search(r'(\d+)시간의\s*.{0,10}치료', text)
    if treatment_match:
        features['treatment_hours'] = int(treatment_match.group(1))
    
    # 취업제한
    if '취업제한' in text:
        features['employment_restriction'] = True
    
    return features

# 테스트
ruling_features = [extract_ruling_features(sample['ruling']) for sample in ds]

print("Ruling 추가 정보 통계:")
print(f"  집행유예 비율: {sum(f['has_suspension'] for f in ruling_features) / len(ruling_features) * 100:.1f}%")
print(f"  사회봉사 명령 비율: {sum(1 for f in ruling_features if f['social_service_hours'] > 0) / len(ruling_features) * 100:.1f}%")
print(f"  치료강의 명령 비율: {sum(1 for f in ruling_features if f['treatment_hours'] > 0) / len(ruling_features) * 100:.1f}%")
print(f"  취업제한 비율: {sum(f['employment_restriction'] for f in ruling_features) / len(ruling_features) * 100:.1f}%")

# 예시
print("\n예시 (첫 5개):")
for i in range(5):
    print(f"\n샘플 {i+1}:")
    print(f"  원문: {ds[i]['ruling']['text'][:100]}...")
    print(f"  추출: {ruling_features[i]}")

---

## 4. Reason 컬럼 전처리

### 📊 현재 상태
- 평균 길이: 299.3자
- 범위: 9자 ~ 7,904자 (매우 큰 편차)
- 내용: 양형 이유 ("양형의 이유, 법률상 처단형의 범위, 양형기준...")

### 🤔 왜 이렇게 전처리할까요?

**Reason의 특성:**
- **복잡한 구조**: 여러 섹션으로 구성 (법률상 처단형, 양형기준, 선고형 결정 이유)
- **핵심 정보**: 가중/감경 사유가 포함되어 있어 **Label 예측에 도움**이 될 수 있습니다
- **길이 편차**: 9자는 거의 비어있음, 7,904자는 매우 상세한 설명

**사용 시나리오별 전처리:**

### 시나리오 1: Reason을 사용하지 않음
- **이유**: Reason은 판결 "결과"를 설명하는 것이므로, 예측 시점에 알 수 없는 정보입니다
- **문제**: 데이터 누수 위험 (Reason에 Label이 언급되기도 함)
- **권장**: 일반적인 Legal Judgment Prediction에서는 사용 안 함

### 시나리오 2: 가중/감경 사유만 추출하여 보조 특성으로 활용
- **이유**: "초범", "자백", "피해 회복", "전과" 등의 키워드는 Facts에 없을 수 있습니다
- **방법**: 키워드 기반으로 감경/가중 요소를 이진 특성으로 추출
- **주의**: 이것도 일종의 데이터 누수일 수 있으므로, 실제 적용시 검증 필요

### 시나리오 3: 텍스트 요약 후 Facts와 결합
- **이유**: Reason의 핵심 내용만 추출하여 Facts를 보강
- **방법**: 문장 단위로 split 후 중요 문장만 선택

In [ ]:
# 시나리오 2: 가중/감경 사유 추출 (신중하게 사용)

def extract_sentencing_factors(reason_text):
    """
    Reason에서 양형 요소 추출
    
    주의사항:
    - 이 정보는 판결 후 작성된 것이므로 "진짜" 예측 시스템에서는 사용하면 안됩니다!
    - 다만, 학습/연구 목적으로 "어떤 요소가 형량에 영향을 주는지" 분석할 때는 유용합니다
    - 실제 시스템 구축시 이런 요소들을 Facts에서 직접 추출하도록 만들어야 합니다
    """
    factors = {
        # 감경 요소
        'first_offense': False,  # 초범
        'confession': False,     # 자백
        'victim_agreement': False,  # 피해자 합의
        'no_punishment_desired': False,  # 처벌불원
        'deep_reflection': False,  # 반성
        'damage_recovery': False,  # 피해 회복
        
        # 가중 요소
        'previous_convictions': False,  # 전과
        'serious_damage': False,  # 중한 피해
        'no_agreement': False,  # 합의 없음
    }
    
    if not reason_text:
        return factors
    
    # 감경 요소 검색
    if any(kw in reason_text for kw in ['초범', '처음']):
        factors['first_offense'] = True
    
    if any(kw in reason_text for kw in ['자백', '범행을 인정']):
        factors['confession'] = True
    
    if any(kw in reason_text for kw in ['합의', '원만히 해결']):
        factors['victim_agreement'] = True
    
    if '처벌불원' in reason_text:
        factors['no_punishment_desired'] = True
    
    if any(kw in reason_text for kw in ['반성', '뉘우침', '참회']):
        factors['deep_reflection'] = True
    
    if any(kw in reason_text for kw in ['피해 회복', '변제', '배상']):
        factors['damage_recovery'] = True
    
    # 가중 요소 검색
    if any(kw in reason_text for kw in ['전과', '동종 전과', '범죄경력']):
        factors['previous_convictions'] = True
    
    if any(kw in reason_text for kw in ['중한 피해', '상당한 피해', '심각한']):
        factors['serious_damage'] = True
    
    if '합의하지 못' in reason_text or '합의가 이루어지지' in reason_text:
        factors['no_agreement'] = True
    
    return factors

# 테스트
reason_factors = [extract_sentencing_factors(sample['reason']) for sample in ds]

print("양형 요소 통계:")
print("\n[감경 요소]")
print(f"  초범: {sum(f['first_offense'] for f in reason_factors)} / {len(ds)} ({sum(f['first_offense'] for f in reason_factors)/len(ds)*100:.1f}%)")
print(f"  자백: {sum(f['confession'] for f in reason_factors)} / {len(ds)} ({sum(f['confession'] for f in reason_factors)/len(ds)*100:.1f}%)")
print(f"  피해자 합의: {sum(f['victim_agreement'] for f in reason_factors)} / {len(ds)} ({sum(f['victim_agreement'] for f in reason_factors)/len(ds)*100:.1f}%)")
print(f"  처벌불원: {sum(f['no_punishment_desired'] for f in reason_factors)} / {len(ds)} ({sum(f['no_punishment_desired'] for f in reason_factors)/len(ds)*100:.1f}%)")

print("\n[가중 요소]")
print(f"  전과: {sum(f['previous_convictions'] for f in reason_factors)} / {len(ds)} ({sum(f['previous_convictions'] for f in reason_factors)/len(ds)*100:.1f}%)")

# 감경/가중 요소 개수에 따른 형량 분석
print("\n감경 요소 개수별 평균 형량 (징역형만, 개월 단위):")
for n_factors in range(5):
    # 해당 개수의 감경 요소를 가진 샘플 필터
    matching = [
        (sample, factors) 
        for sample, factors in zip(ds, reason_factors) 
        if sum([
            factors['first_offense'],
            factors['confession'],
            factors['victim_agreement'],
            factors['no_punishment_desired']
        ]) == n_factors and '징역' in sample['label']['text'] and '월' in sample['label']['text']
    ]
    
    if matching:
        months_list = []
        for sample, _ in matching:
            text = sample['label']['text']
            months_match = re.search(r'(\d+)월', text)
            if months_match:
                months_list.append(int(months_match.group(1)))
        
        if months_list:
            avg_months = sum(months_list) / len(months_list)
            print(f"  감경 요소 {n_factors}개: 평균 {avg_months:.1f}개월 (샘플 {len(matching)}개)")

In [ ]:
# 시나리오 3: Reason 요약 (선택적)

def summarize_reason(reason_text, max_sentences=3):
    """
    Reason의 핵심 문장만 추출
    
    전략:
    1. 정형화된 부분("법률상 처단형의 범위", "양형기준" 등) 제거
    2. 실질적인 양형 이유가 담긴 문장만 선택
    3. 최대 N개 문장으로 제한
    
    왜 이렇게 할까?
    - Reason은 길고 반복적인 내용이 많습니다
    - 핵심만 추출하면 모델이 집중할 포인트가 명확해집니다
    """
    if not reason_text or len(reason_text) < 10:
        return ""
    
    # 문장 단위로 분리
    sentences = re.split(r'[.\n]', reason_text)
    
    # 필터링: 정형 문구 제거
    skip_keywords = [
        '양형의 이유',
        '법률상 처단형의 범위',
        '양형기준에 따른',
        '유형의 결정',
        '권고영역',
        '선고형의 결정'
    ]
    
    useful_sentences = []
    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 10:  # 너무 짧은 문장 제외
            continue
        if any(kw in sent for kw in skip_keywords):  # 정형 문구 제외
            continue
        useful_sentences.append(sent)
    
    # 최대 N개 문장 선택
    selected = useful_sentences[:max_sentences]
    
    return ' '.join(selected)

# 테스트
print("Reason 요약 예시:\n")
for i in range(3):
    original = ds[i]['reason']
    summarized = summarize_reason(original)
    
    print(f"--- 샘플 {i+1} ---")
    print(f"원본 ({len(original)}자):")
    print(original[:200] + "..." if len(original) > 200 else original)
    print(f"\n요약 ({len(summarized)}자):")
    print(summarized)
    print()

---

## 5. 종합 전처리 파이프라인

### 🎯 추천 전처리 전략

**목적: Facts → Label 예측 (Legal Judgment Prediction)**

#### 사용할 컬럼:
1. **Facts (입력)**: 전처리 후 사용
2. **Label (출력)**: 범주화하여 사용 (전략 A 권장)
3. **Ruling**: 집행유예 정보만 추가 레이블로 사용 (선택)
4. **Reason**: ⚠️ 사용 안 함 (데이터 누수 위험)

#### 전처리 순서:
```
1. Facts 전처리 (정규화, 익명화, 길이 제한)
2. Label 범주화 (73개 → 10~15개)
3. (선택) Ruling에서 집행유예 여부 추출
4. Train/Valid/Test 분할 확인
5. 토크나이저 적용
```

In [ ]:
# 종합 전처리 함수

def preprocess_dataset(dataset, use_suspension=False):
    """
    전체 데이터셋 전처리
    
    Args:
        dataset: HuggingFace Dataset
        use_suspension: 집행유예 정보를 추가 레이블로 사용할지 여부
    
    Returns:
        처리된 데이터셋 (pandas DataFrame)
    """
    processed_data = []
    
    for sample in dataset:
        # Facts 전처리
        facts_clean = preprocess_facts(sample['facts'])
        
        # Label 범주화
        label_category = label_to_category(sample['label'])
        
        # 기본 데이터
        processed = {
            'id': sample['id'],
            'casetype': sample['casetype'],
            'casename': sample['casename'],
            'facts_original': sample['facts'],
            'facts_clean': facts_clean,
            'label_original': sample['label']['text'],
            'label_category': label_category,
        }
        
        # 집행유예 정보 추가 (선택)
        if use_suspension:
            ruling_features = extract_ruling_features(sample['ruling'])
            processed['has_suspension'] = ruling_features['has_suspension']
        
        processed_data.append(processed)
    
    return pd.DataFrame(processed_data)

# 실행
print("전처리 중...")
df_processed = preprocess_dataset(ds, use_suspension=True)

print("\n전처리 완료!")
print(f"총 샘플 수: {len(df_processed)}")
print(f"\n컬럼 목록:")
print(df_processed.columns.tolist())

print("\n범주화된 라벨 분포:")
print(df_processed['label_category'].value_counts())

print("\n집행유예 비율:")
print(df_processed['has_suspension'].value_counts(normalize=True))

# 샘플 확인
print("\n샘플 1개:")
print(df_processed.iloc[0].to_dict())

In [ ]:
# 저장
df_processed.to_csv('ljp_criminal_preprocessed.csv', index=False, encoding='utf-8-sig')
print("저장 완료: ljp_criminal_preprocessed.csv")

---

## 6. 모델 학습을 위한 추가 전처리

### 토크나이징 및 데이터 로더 준비

In [ ]:
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder

# 한국어 BERT 토크나이저
tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')

# 라벨 인코딩
label_encoder = LabelEncoder()
df_processed['label_encoded'] = label_encoder.fit_transform(df_processed['label_category'])

print("라벨 매핑:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i}: {label}")

# 토크나이징 예시
sample_text = df_processed['facts_clean'].iloc[0]
tokens = tokenizer(sample_text, max_length=512, truncation=True, padding='max_length', return_tensors='pt')

print(f"\n토큰화 결과:")
print(f"  Input IDs shape: {tokens['input_ids'].shape}")
print(f"  Attention mask shape: {tokens['attention_mask'].shape}")

---

## 7. 요약 및 권장사항

### ✅ 권장 전처리 방법

| 컬럼 | 사용 여부 | 전처리 방법 | 이유 |
|------|----------|-------------|------|
| **Facts** | ✅ 사용 | 날짜/시간/인명 정규화, 길이 제한 | 입력 데이터, 토큰 효율성 향상 |
| **Label** | ✅ 사용 | 범주화 (73 → 10~15개) | 출력 데이터, 클래스 불균형 완화 |
| **Ruling** | ⚠️ 선택 | 집행유예 정보만 추출 | 추가 레이블로 활용 가능 |
| **Reason** | ❌ 미사용 | - | 데이터 누수 위험 |

### 📊 기대 효과

1. **클래스 수 감소**: 73개 → 10~15개 (학습 안정성 ↑)
2. **토큰 수 감소**: 정규화로 불필요한 토큰 제거
3. **일반화 능력 향상**: 과적합 방지
4. **실용성**: 실무에서 유의미한 구간 예측

### 🚀 다음 단계

1. 전처리된 데이터로 BERT 계열 모델 학습
2. 교차 검증으로 성능 평가
3. 혼동 행렬(Confusion Matrix)로 어떤 라벨이 헷갈리는지 분석
4. 오분류 사례 분석하여 전처리 개선